# Демонстрация пакета `company_capitalization`

Ноутбук не копирует функции: всё делается вызовами установленного пакета.
Запуск: `poetry install --with dev,notebook` и затем Jupyter/VS Code с ядром из `.venv`.
Рабочая директория — корень проекта.

In [ ]:
from pathlib import Path
import os

# Allow running from notebooks/ or from the project root.
if Path.cwd().name == "notebooks":
    os.chdir(Path.cwd().parent)

import pandas as pd

from company_capitalization.data import load_training_data
from company_capitalization.features import build_features, week52_diagnostics
from company_capitalization.training import TrainConfig, train
from company_capitalization.prediction import load_model, predict_frame

## Данные: проверка схемы и правила отбора строк

In [ ]:
X, y, info = load_training_data(Path("financials_src.csv"))
print(info["sha256"])
print(info["row_selection"])
print(week52_diagnostics(X))

## Признаки строятся только из входов (без `Market Cap`)

In [ ]:
build_features(X).head()

## Быстрый baseline: Ridge, CV на train, отдельный holdout

In [ ]:
result = train(TrainConfig(data_path=Path("financials_src.csv"),
                           output_dir=Path("artifacts/demo_ridge"), model="ridge", seed=42))
pd.DataFrame({"holdout": result.metrics["holdout"]})

## Прогноз для таблицы без целевой колонки

In [ ]:
model = load_model(Path("artifacts/demo_ridge"))  # only trusted, self-made artifacts
new = pd.read_csv("examples/companies_without_target.csv")
predict_frame(model, new).head()